# Analysebeispiele

Dieses Notebook zeigt anhand ausgewählter Beispiele, wie die durch die Pipeline aufbereiteten und transformierten Daten für weiterführende Analysen
verwendet werden können.

Der Schwerpunkt liegt dabei nicht auf einer vollständigen Analyse des Datensatzes, sondern darauf, die praktische Nutzbarkeit der erzeugten Tabellen
und der während der Transformation vorbereiteten Merkmale zu demonstrieren.

## Vorbereitung

Für die Analysebeispiele werden die während der Transformation erzeugten Parquet-Dateien geladen.

In [ ]:
from src.paths import TRANSFORMED_DATA_DIR, IMAGES_DIR
from matplotlib.ticker import EngFormatter
import matplotlib.pyplot as plt
import pandas as pd

date_dataset = pd.read_parquet(TRANSFORMED_DATA_DIR / "date.parquet")

invoice_dataset = pd.read_parquet(TRANSFORMED_DATA_DIR / "invoice.parquet")

invoice_position_dataset = pd.read_parquet(TRANSFORMED_DATA_DIR / "invoice_position.parquet")

Eine globale Matplotlib-Konfiguration wire definiert, um für die folgenden Visualisierungen eine einheitliche Darstellung zu verwenden.

In [ ]:
plt.style.use("default")

plt.rcParams.update({
    "figure.figsize": (16, 8),
    "font.size": 12,
    "axes.titlesize": 18,
    "axes.titleweight": "bold",
    "axes.titlepad": 20,
    "axes.labelsize": 14,
    "axes.labelweight": "bold",
    "axes.labelpad": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 100,
})

## Monatliche Entwicklung zentraler Kennzahlen

Als erstes Beispiel wird die zeitliche Entwicklung von Umsatz, Absatz, Anzahl der Bestellungen und durchschnittlichem Bestellwert betrachtet.

Dazu wird die Rechnungstabelle mit der Datumstabelle verknüpft. Über das Rechnungsdatum können dadurch die vorbereiteten Kalendermerkmale der
Datumstabelle, insbesondere `YearMonth`, direkt für die monatliche Gruppierung verwendet werden.

Da der letzte im Datensatz enthaltene Monat nicht vollständig vorliegt, wird dieser vor der Aggregation ausgeschlossen.

In [ ]:
invoice_date_merge = pd.merge(
    invoice_dataset,
    date_dataset,
    left_on="Date",
    right_index=True,
    how="inner"
)

incomplete_last_month = invoice_date_merge["YearMonth"].max()

invoice_date_merge = invoice_date_merge[invoice_date_merge["YearMonth"] < incomplete_last_month]

invoice_date_merge_group = invoice_date_merge.groupby("YearMonth")

revenue_by_yearmonth = invoice_date_merge_group["TotalRevenue"].sum()

width, height = plt.rcParams["figure.figsize"]

fig, axis = plt.subplots(4, 1, sharex=True, figsize=(width, height * 4))

revenue_by_yearmonth.plot(
    ax=axis[0],
    title="Monthly Revenue",
    xlabel="Month",
    ylabel="Revenue",
    color="blue",
    marker="o",
)

quantity_by_yearmonth = invoice_date_merge_group["TotalQuantity"].sum()

quantity_by_yearmonth.plot(
    ax=axis[1],
    title="Monthly Quantity",
    xlabel="Month",
    ylabel="Quantity",
    color="green",
    marker="o",
)

monthly_order_count = invoice_date_merge_group.size()

monthly_order_count.plot(
    ax=axis[2],
    title="Monthly Order Count",
    xlabel="Month",
    ylabel="Order Count",
    color="red",
    marker="o",
)

average_order_value = revenue_by_yearmonth / monthly_order_count

average_order_value.plot(
    ax=axis[3],
    title="Monthly Average Order Value",
    xlabel="Month",
    ylabel="Average Order Value",
    color="black",
    marker="o",
)

for ax in axis:
    ax.yaxis.set_major_formatter(EngFormatter())
    ax.set_xlim(right=revenue_by_yearmonth.index.max() + pd.Timedelta(days=30))

fig.subplots_adjust(hspace=0.2)

fig.savefig(IMAGES_DIR / "monthly_business_metrics.png", bbox_inches="tight")

![Monthly Business Metrics](../images/monthly_business_metrics.png)